# Act 4 - The MLOps loop: add a feature, retrain, compare

The data scientist iterates entirely in the **dev sandbox** (`ML_DEV_ROLE`):
1. register a **new feature view** (`ACCOUNT_RISK`),
2. **retrain** the model (V2) with the added signals,
3. **log** the run to the experiment for side-by-side comparison, and
4. **register** V2 to the dev registry and compare V1 -> V2.

Nothing here can touch production - that's the promotion gate (Act 5).

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("USE ROLE ML_DEV_ROLE").collect()
session.sql("USE WAREHOUSE CORTEX_CODE_WH").collect()

# --- constants (self-contained so the notebook needs no repo import) ---
DEV_DB, PROD_DB = "ML_FRAUD_DEV_SANDBOX", "ML_FRAUD_PRODUCTION"
FS_SCHEMA, CURATED, REG_SCHEMA, EXP_SCHEMA = "FEATURE_STORE", "CURATED", "ML", "EXPERIMENTS"
KEY, ENTITY, MODEL = "ACCOUNT_ID", "ACCOUNT", "AML_FRAUD_GBM"
FV_PROFILE, RISK_FV, FV_VER = "ACCOUNT_PROFILE", "ACCOUNT_RISK", "V1"
NEG, NEW_VERSION, EXPERIMENT = 400_000, "V2", "AML_FRAUD_TRAINING"
ABT = f"{DEV_DB}.{CURATED}.FRAUD_ABT"
ACCT_HIST = f"{PROD_DB}.{CURATED}.ACCOUNT_HISTORY"

session.sql(f"USE SCHEMA {DEV_DB}.{FS_SCHEMA}").collect()
print("role:", session.get_current_role())

## 1. Add a new feature view - `ACCOUNT_RISK`\n\nEngineered risk signals: spend volatility and receiver fan-out (the classic mule tell).

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

fs = FeatureStore(session=session, database=DEV_DB, name=FS_SCHEMA,
                  default_warehouse="CORTEX_CODE_WH", creation_mode=CreationMode.CREATE_IF_NOT_EXIST)
account = Entity(name=ENTITY, join_keys=[KEY]); fs.register_entity(account)

risk_df = session.sql(f'''
    SELECT {KEY},
           HIST_STD_AMOUNT / NULLIF(HIST_AVG_AMOUNT,0)        AS HIST_AMOUNT_CV,
           HIST_DISTINCT_RECEIVERS / NULLIF(HIST_TXN_COUNT,0) AS HIST_RECEIVER_FANOUT
    FROM {ACCT_HIST}''')
fs.register_feature_view(
    FeatureView(name=RISK_FV, entities=[account], feature_df=risk_df, refresh_freq="1 day",
                desc="Engineered risk: spend volatility + receiver fan-out."),
    version=FV_VER, overwrite=True)
print("registered", RISK_FV, FV_VER)

## 2. Retrain V2 using the feature store (profile + risk)

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

EXCLUDE = {KEY, "EVENT_TS", "SPLIT", "IS_LAUNDERING", "PAYMENT_CURRENCY", "RECEIVING_CURRENCY", "TO_COUNTRY"}
MONO_POS = {"AMOUNT_PAID","IS_CROSS_CURRENCY","IS_CROSS_BORDER","IS_HIGH_RISK_FORMAT",
            "AMOUNT_TO_AVG_RATIO","HIST_RECEIVER_FANOUT"}

def recall_at_k(y, s, k=0.01):
    n = max(1, int(len(s)*k)); idx = np.argsort(s)[::-1][:n]
    return y[idx].sum()/y.sum() if y.sum() else 0.0

# downsample negatives (keep all positives), preserving SPLIT
sample = f"{DEV_DB}.{CURATED}.FRAUD_ABT_SAMPLE"
session.sql(f'''CREATE OR REPLACE TABLE {sample} AS
    SELECT * FROM {ABT} WHERE IS_LAUNDERING=1
    UNION ALL SELECT * FROM (SELECT * FROM {ABT} WHERE IS_LAUNDERING=0) SAMPLE ({NEG} ROWS)''').collect()

profile = fs.get_feature_view(FV_PROFILE, FV_VER); risk = fs.get_feature_view(RISK_FV, FV_VER)
pdf = fs.generate_training_set(spine_df=session.table(sample), features=[profile, risk],
                               spine_label_cols=["IS_LAUNDERING"]).to_pandas()
pdf.columns = [c.upper() for c in pdf.columns]

y = pdf["IS_LAUNDERING"].astype(int).values; split = pdf["SPLIT"].values
cat = [c for c in ["PAYMENT_FORMAT"] if c in pdf.columns]
num = [c for c in pdf.columns if c not in EXCLUDE and c not in cat]
X = pd.concat([pdf[num].apply(pd.to_numeric, errors="coerce").fillna(0.0),
               pd.get_dummies(pdf[cat].astype(str), prefix=cat)], axis=1)
X.columns = [str(c).upper().replace(" ","_") for c in X.columns]
tr, te = split=="TRAIN", split=="TEST"
spw = (~y[tr].astype(bool)).sum()/max(1, y[tr].sum())
mono = [1 if c in MONO_POS else 0 for c in X.columns]
params = dict(max_iter=300, max_depth=6, learning_rate=0.1, l2_regularization=1.0, min_samples_leaf=50, random_state=42)

model = HistGradientBoostingClassifier(monotonic_cst=mono, **params)
model.fit(X[tr], y[tr], sample_weight=np.where(y[tr]==1, spw, 1.0))
proba = model.predict_proba(X[te])[:,1]
metrics = {"pr_auc": float(average_precision_score(y[te], proba)),
           "roc_auc": float(roc_auc_score(y[te], proba)),
           "recall_at_1pct": float(recall_at_k(y[te], proba))}
print("features:", X.shape[1], "(added HIST_AMOUNT_CV, HIST_RECEIVER_FANOUT)")
print("V2 TEST metrics:", {k: round(v,4) for k,v in metrics.items()})

## 3. Log the experiment run (params + metrics)

In [ ]:
from snowflake.ml.experiment import ExperimentTracking
exp = ExperimentTracking(session=session, database_name=DEV_DB, schema_name=EXP_SCHEMA)
exp.set_experiment(EXPERIMENT)
run = f"train_{NEW_VERSION}"
try:
    exp.delete_run(run)
except Exception:
    pass
with exp.start_run(run):
    exp.log_params({**params, "scale_pos_weight": round(spw,2), "n_features": X.shape[1],
                    "added_features": "HIST_AMOUNT_CV,HIST_RECEIVER_FANOUT"})
    exp.log_metrics(metrics)
print("logged run", run, "to experiment", EXPERIMENT)

## 4. Register V2 to the dev registry and compare to V1

In [ ]:
from snowflake.ml.registry import Registry
reg = Registry(session=session, database_name=DEV_DB, schema_name=REG_SCHEMA)
try:
    reg.get_model(MODEL).delete_version(NEW_VERSION)   # idempotent re-run
except Exception:
    pass
reg.log_model(model=model, model_name=MODEL, version_name=NEW_VERSION,
              sample_input_data=X[te].head(20), metrics=metrics,
              comment="V2: added engineered risk features (CV, receiver fan-out).")
try:
    v1 = reg.get_model(MODEL).version("V1").show_metrics()
except Exception:
    v1 = {}
print("metric        V1        V2        delta")
for k in ("pr_auc","roc_auc","recall_at_1pct"):
    a, b = float(v1.get(k, float("nan"))), metrics[k]
    print(f"{k:<13} {a:>7.4f}  {b:>7.4f}  {(b-a):>+7.4f}")
reg.get_model(MODEL).default = NEW_VERSION
print(f"\n{MODEL}/{NEW_VERSION} registered in dev; set as default (candidate for promotion).")

## Done - V2 is the candidate

V2 beats V1 (higher PR-AUC and recall@1%). It's the dev default now, but a data scientist
**cannot** deploy it. Promotion to production happens only through the gated GitHub Actions
pipeline (Act 5).